In [1]:
import torch
import gc
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

class ModelRegistry:
    def __init__(self, max_vram_mb=5000):
        self._cache = {}  # key: model_name -> {"model": ..., "tokenizer": ...}
        self.max_vram_mb = max_vram_mb  # safety ceiling, leaves headroom below your 6GB card

    def _current_vram_mb(self):
        return torch.cuda.memory_reserved() / 1024**2

    def _evict_oldest(self):
        """Remove the least-recently-used model to free VRAM."""
        if not self._cache:
            return
        oldest_key = next(iter(self._cache))  # dict preserves insertion order in Python 3.7+
        print(f"Evicting '{oldest_key}' to free VRAM...")
        del self._cache[oldest_key]
        gc.collect()
        torch.cuda.empty_cache()

    def get(self, name, loader_fn):
        """
        name: a string key like 'sentiment' or 'summarizer'
        loader_fn: a function that loads and returns the model+tokenizer when called
        """
        if name in self._cache:
            # move to end to mark as "recently used"
            self._cache[name] = self._cache.pop(name)
            return self._cache[name]

        # Evict if we're near the VRAM ceiling before loading something new
        while self._current_vram_mb() > self.max_vram_mb and self._cache:
            self._evict_oldest()

        print(f"Loading '{name}' for the first time...")
        loaded = loader_fn()
        self._cache[name] = loaded
        return loaded

registry = ModelRegistry(max_vram_mb=5000)

In [2]:
def load_sentiment_model():
    from transformers import pipeline
    return pipeline("sentiment-analysis", device=0)  # device=0 means first GPU

def load_embedding_model():
    return SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

def load_summarizer():
    tok = AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")
    mdl = AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6").to("cuda")
    return {"tokenizer": tok, "model": mdl}

In [3]:
import time

# First call: should print "Loading 'sentiment' for the first time..." and take a few seconds
start = time.time()
sentiment_pipe = registry.get("sentiment", load_sentiment_model)
print(f"First call took {time.time() - start:.2f}s")

# Second call: should NOT reload — should be near-instant
start = time.time()
sentiment_pipe_again = registry.get("sentiment", load_sentiment_model)
print(f"Second call took {time.time() - start:.4f}s")

print("Same object?", sentiment_pipe is sentiment_pipe_again)

Loading 'sentiment' for the first time...


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

First call took 2.47s
Second call took 0.0000s
Same object? True


In [4]:
embed_model = registry.get("embedding", load_embedding_model)
summarizer = registry.get("summarizer", load_summarizer)

print("Currently cached models:", list(registry._cache.keys()))
print(f"Current VRAM reserved: {registry._current_vram_mb():.0f} MB")

Loading 'embedding' for the first time...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading 'summarizer' for the first time...


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Currently cached models: ['sentiment', 'embedding', 'summarizer']
Current VRAM reserved: 1544 MB


In [5]:
def load_qwen():
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    mdl = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16
    ).to("cuda")
    return {"tokenizer": tok, "model": mdl}

qwen_bundle = registry.get("qwen", load_qwen)

print("Currently cached models:", list(registry._cache.keys()))
print(f"Current VRAM reserved: {registry._current_vram_mb():.0f} MB")

Loading 'qwen' for the first time...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Currently cached models: ['sentiment', 'embedding', 'summarizer', 'qwen']
Current VRAM reserved: 4678 MB


In [6]:
def load_translator():
    from transformers import MarianTokenizer, MarianMTModel
    tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
    mdl = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-hi").to("cuda")
    return {"tokenizer": tok, "model": mdl}

translator = registry.get("translator", load_translator)

print("Currently cached models:", list(registry._cache.keys()))
print(f"Current VRAM reserved: {registry._current_vram_mb():.0f} MB")

Loading 'translator' for the first time...


c:\Projects\nlp-toolkit\hf-toolkit-env\lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Currently cached models: ['sentiment', 'embedding', 'summarizer', 'qwen', 'translator']
Current VRAM reserved: 4974 MB


In [7]:
def load_bert():
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    mdl = AutoModel.from_pretrained("bert-base-uncased").to("cuda")
    return {"tokenizer": tok, "model": mdl}

bert_bundle = registry.get("bert_raw", load_bert)

print("Currently cached models:", list(registry._cache.keys()))
print(f"Current VRAM reserved: {registry._current_vram_mb():.0f} MB")

Loading 'bert_raw' for the first time...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Currently cached models: ['sentiment', 'embedding', 'summarizer', 'qwen', 'translator', 'bert_raw']
Current VRAM reserved: 5424 MB


In [8]:
class ModelRegistry:
    def __init__(self, max_vram_mb=5000):
        self._cache = {}
        self.max_vram_mb = max_vram_mb

    def _current_vram_mb(self):
        return torch.cuda.memory_reserved() / 1024**2

    def _evict_oldest(self, protect_key=None):
        """Remove the least-recently-used model, skipping the one we just loaded."""
        for key in list(self._cache.keys()):
            if key == protect_key:
                continue
            print(f"Evicting '{key}' to free VRAM...")
            del self._cache[key]
            gc.collect()
            torch.cuda.empty_cache()
            return True
        return False

    def get(self, name, loader_fn):
        if name in self._cache:
            self._cache[name] = self._cache.pop(name)
            return self._cache[name]

        print(f"Loading '{name}' for the first time...")
        loaded = loader_fn()
        self._cache[name] = loaded

        # NOW check AFTER loading — evict other models if we've gone over budget
        while self._current_vram_mb() > self.max_vram_mb:
            evicted_something = self._evict_oldest(protect_key=name)
            if not evicted_something:
                print("Warning: over VRAM budget but nothing left to evict!")
                break

        return loaded

registry = ModelRegistry(max_vram_mb=5000)

In [9]:
del sentiment_pipe, sentiment_pipe_again, embed_model, summarizer, qwen_bundle, translator, bert_bundle
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after cleanup: {registry._current_vram_mb():.0f} MB")

VRAM after cleanup: 0 MB


In [10]:
sentiment_pipe = registry.get("sentiment", load_sentiment_model)
print(f"After sentiment: {registry._current_vram_mb():.0f} MB, cached: {list(registry._cache.keys())}")

embed_model = registry.get("embedding", load_embedding_model)
print(f"After embedding: {registry._current_vram_mb():.0f} MB, cached: {list(registry._cache.keys())}")

summarizer = registry.get("summarizer", load_summarizer)
print(f"After summarizer: {registry._current_vram_mb():.0f} MB, cached: {list(registry._cache.keys())}")

qwen_bundle = registry.get("qwen", load_qwen)
print(f"After qwen: {registry._current_vram_mb():.0f} MB, cached: {list(registry._cache.keys())}")

translator = registry.get("translator", load_translator)
print(f"After translator: {registry._current_vram_mb():.0f} MB, cached: {list(registry._cache.keys())}")

bert_bundle = registry.get("bert_raw", load_bert)
print(f"After bert_raw: {registry._current_vram_mb():.0f} MB, cached: {list(registry._cache.keys())}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading 'sentiment' for the first time...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

After sentiment: 292 MB, cached: ['sentiment']
Loading 'embedding' for the first time...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

After translator: 4974 MB, cached: ['sentiment', 'embedding', 'summarizer', 'qwen', 'translator']
Loading 'bert_raw' for the first time...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evicting 'sentiment' to free VRAM...
Evicting 'embedding' to free VRAM...
Evicting 'summarizer' to free VRAM...
Evicting 'qwen' to free VRAM...
Evicting 'translator' to free VRAM...
After bert_raw: 5424 MB, cached: ['bert_raw']


In [11]:
del sentiment_pipe, embed_model, summarizer, qwen_bundle, translator
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after clearing dangling references: {registry._current_vram_mb():.0f} MB")
print(f"Still cached in registry: {list(registry._cache.keys())}")


VRAM after clearing dangling references: 734 MB
Still cached in registry: ['bert_raw']
